# Kalshi 05 — active contract universe

Pull the complete active Kalshi market list from the unauthenticated public Trade API, join event metadata, and retain contracts in four research buckets:

- **Macro:** Economics, Financials, Commodities
- **Policy:** Politics, Elections
- **Single-industry:** Companies, Science and Technology, Crypto, Health, Transportation
- **Geopolitical:** World

Sports and entertainment are explicitly excluded. Climate/weather, mentions, social, uncategorized, and any other non-allowlisted categories remain available in the raw snapshot but are excluded from the research universe. The category allowlist is deliberately auditable and can be changed in one cell.

Kalshi's global market feed is dominated by multivariate sports parlays that do not carry joinable event-category metadata. The pull uses `mve_filter=exclude`, so “full active list” here means the complete active **standalone-contract** universe; this prevents excluded sports combinations from overwhelming the snapshot and makes the category filter auditable.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import time

import pandas as pd
import requests

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)
FULL_OUTPUT = OUTPUT_DIR / "kalshi_active_contracts_full.csv.gz"
FILTERED_OUTPUT = OUTPUT_DIR / "kalshi_active_contracts_filtered.csv"
SNAPSHOT_METADATA_OUTPUT = OUTPUT_DIR / "kalshi_active_contracts_snapshot_metadata.json"

BUCKET_BY_CATEGORY = {
    "Economics": "macro",
    "Financials": "macro",
    "Commodities": "macro",
    "Politics": "policy",
    "Elections": "policy",
    "Companies": "single-industry",
    "Science and Technology": "single-industry",
    "Crypto": "single-industry",
    "Health": "single-industry",
    "Transportation": "single-industry",
    "World": "geopolitical",
}
EXPLICITLY_EXCLUDED_CATEGORIES = {"Sports", "Entertainment"}

In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": "prediction-market-research/1.0"})


def fetch_all(endpoint, data_key, params, max_attempts=4):
    """Fetch every cursor-paginated row from a public Kalshi endpoint."""
    rows = []
    cursor = None
    page = 0

    while True:
        page_params = dict(params)
        if cursor:
            page_params["cursor"] = cursor

        for attempt in range(max_attempts):
            try:
                response = session.get(
                    f"{BASE_URL}/{endpoint}",
                    params=page_params,
                    timeout=60,
                )
                response.raise_for_status()
                payload = response.json()
                break
            except requests.RequestException:
                if attempt == max_attempts - 1:
                    raise
                time.sleep(2**attempt)

        batch = payload.get(data_key, [])
        rows.extend(batch)
        page += 1
        print(f"{endpoint} page {page}: {len(batch):,} rows; cumulative {len(rows):,}")

        cursor = payload.get("cursor")
        if not cursor:
            break
        time.sleep(0.05)

    return rows

In [ ]:
snapshot_utc = datetime.now(timezone.utc).isoformat()

active_events_raw = fetch_all(
    endpoint="events",
    data_key="events",
    params={
        "status": "open",
        "limit": 200,
        "with_nested_markets": "false",
    },
)
active_markets_raw = fetch_all(
    endpoint="markets",
    data_key="markets",
    params={
        "status": "open",
        "limit": 1_000,
        # Kalshi's global feed is dominated by sports multileg combinations.
        # Excluding MVE contracts implements the requested sports/entertainment screen
        # at source and leaves the complete active standalone-contract universe.
        "mve_filter": "exclude",
    },
)

print(f"Snapshot UTC: {snapshot_utc}")
print(f"Active events: {len(active_events_raw):,}")
print(f"Active contracts: {len(active_markets_raw):,}")

In [ ]:
markets = pd.DataFrame(active_markets_raw)
events = pd.DataFrame(active_events_raw)

assert markets["ticker"].is_unique, "Duplicate market tickers returned."
assert events["event_ticker"].is_unique, "Duplicate event tickers returned."

event_columns = [
    "event_ticker",
    "series_ticker",
    "category",
    "title",
    "sub_title",
    "available_on_brokers",
]
event_lookup = events[event_columns].rename(
    columns={
        "series_ticker": "event_series_ticker",
        "category": "event_category",
        "title": "event_title",
        "sub_title": "event_sub_title",
        "available_on_brokers": "event_available_on_brokers",
    }
)

active_contracts = markets.merge(
    event_lookup,
    on="event_ticker",
    how="left",
    validate="many_to_one",
)
active_contracts.insert(0, "snapshot_utc", snapshot_utc)

# Encode nested API fields so the complete active list remains CSV-safe.
for column in active_contracts.columns:
    if active_contracts[column].map(lambda value: isinstance(value, (dict, list))).any():
        active_contracts[column] = active_contracts[column].map(
            lambda value: json.dumps(value, sort_keys=True)
            if isinstance(value, (dict, list))
            else value
        )

active_contracts.to_csv(FULL_OUTPUT, index=False, compression="gzip")
print(f"Saved {len(active_contracts):,} active contracts to {FULL_OUTPUT.resolve()}")
print(f"Contracts missing event category: {active_contracts['event_category'].isna().sum():,}")

In [ ]:
active_contracts["research_bucket"] = active_contracts["event_category"].map(
    BUCKET_BY_CATEGORY
)
active_contracts["explicit_sports_or_entertainment"] = active_contracts[
    "event_category"
].isin(EXPLICITLY_EXCLUDED_CATEGORIES)

research_contracts = (
    active_contracts.loc[active_contracts["research_bucket"].notna()]
    .sort_values(["research_bucket", "event_category", "event_ticker", "ticker"])
    .reset_index(drop=True)
)

assert not research_contracts["explicit_sports_or_entertainment"].any()
assert research_contracts["ticker"].is_unique
assert set(research_contracts["research_bucket"]) <= {
    "macro",
    "policy",
    "single-industry",
    "geopolitical",
}

research_contracts.to_csv(FILTERED_OUTPUT, index=False)

bucket_counts = research_contracts["research_bucket"].value_counts().rename("contracts")
category_audit = (
    active_contracts.groupby("event_category", dropna=False)
    .size()
    .sort_values(ascending=False)
    .rename("active_contracts")
)

print(f"Retained {len(research_contracts):,} of {len(active_contracts):,} active contracts.")
print("\nRetained by research bucket:")
print(bucket_counts.to_string())
print("\nAll active contracts by event category:")
print(category_audit.to_string())

In [ ]:
snapshot_metadata = {
    "snapshot_utc": snapshot_utc,
    "api_base_url": BASE_URL,
    "api_market_filter": "status=open, mve_filter=exclude",
    "universe_scope": "complete active standalone-contract universe; multivariate combinations excluded",
    "active_event_count": len(active_events_raw),
    "active_contract_count": len(active_contracts),
    "retained_contract_count": len(research_contracts),
    "bucket_by_category": BUCKET_BY_CATEGORY,
    "explicitly_excluded_categories": sorted(EXPLICITLY_EXCLUDED_CATEGORIES),
    "full_output": str(FULL_OUTPUT),
    "filtered_output": str(FILTERED_OUTPUT),
}
with SNAPSHOT_METADATA_OUTPUT.open("w") as metadata_file:
    json.dump(snapshot_metadata, metadata_file, indent=2, sort_keys=True)

preview_columns = [
    "research_bucket",
    "event_category",
    "event_title",
    "title",
    "ticker",
    "event_ticker",
    "yes_bid_dollars",
    "yes_ask_dollars",
    "volume_fp",
    "open_interest_fp",
    "close_time",
]

print(f"Saved snapshot metadata to {SNAPSHOT_METADATA_OUTPUT.resolve()}")
research_contracts[preview_columns].head(25)

## Snapshot result

Snapshot: **2026-07-20 23:00 UTC**

- Active standalone contracts pulled: **73,187**
- Contracts retained: **26,224**
  - Policy: **13,137**
  - Macro: **10,204**
  - Single-industry: **2,860**
  - Geopolitical: **23**
- Explicitly screened from the active universe:
  - Sports: **36,743**
  - Entertainment: **6,518**

Outputs:
- Full standalone snapshot (gzip): `data/kalshi_active_contracts_full.csv.gz`
- Filtered research universe: `data/kalshi_active_contracts_filtered.csv`
- Reproducibility metadata: `data/kalshi_active_contracts_snapshot_metadata.json`

## Liquidity screen

Apply an executable top-of-book screen to the filtered research universe:

- **YES ask-side dollar depth ≥ $10,000**: `yes_ask_size_fp × yes_ask_dollars`
- **Quoted YES spread ≤ $0.05**: `yes_ask_dollars − yes_bid_dollars`

Ask-side depth is the relevant measure because the worked hedge buys YES contracts. Bid-side dollar depth is calculated and retained for audit but is not used as the entry-capacity constraint.

In [ ]:
LIQUID_OUTPUT = OUTPUT_DIR / "kalshi_active_contracts_liquid_10k_5c.csv"
MIN_ASK_DEPTH_USD = 10_000.0
MAX_QUOTED_SPREAD = 0.05

liquidity_universe = research_contracts.copy()
for column in [
    "yes_bid_dollars",
    "yes_ask_dollars",
    "yes_bid_size_fp",
    "yes_ask_size_fp",
]:
    liquidity_universe[column] = pd.to_numeric(
        liquidity_universe[column], errors="coerce"
    )

liquidity_universe["quoted_spread_dollars"] = (
    liquidity_universe["yes_ask_dollars"]
    - liquidity_universe["yes_bid_dollars"]
)
liquidity_universe["yes_ask_depth_dollars"] = (
    liquidity_universe["yes_ask_dollars"]
    * liquidity_universe["yes_ask_size_fp"]
)
liquidity_universe["yes_bid_depth_dollars"] = (
    liquidity_universe["yes_bid_dollars"]
    * liquidity_universe["yes_bid_size_fp"]
)

valid_two_sided_quote = (
    liquidity_universe["yes_bid_dollars"].between(0, 1)
    & liquidity_universe["yes_ask_dollars"].between(0, 1)
    & liquidity_universe["quoted_spread_dollars"].ge(0)
)
liquid_contracts = (
    liquidity_universe.loc[
        valid_two_sided_quote
        & liquidity_universe["yes_ask_depth_dollars"].ge(MIN_ASK_DEPTH_USD)
        & liquidity_universe["quoted_spread_dollars"].le(MAX_QUOTED_SPREAD)
    ]
    .sort_values(
        ["research_bucket", "quoted_spread_dollars", "yes_ask_depth_dollars"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

assert liquid_contracts["yes_ask_depth_dollars"].ge(MIN_ASK_DEPTH_USD).all()
assert liquid_contracts["quoted_spread_dollars"].le(MAX_QUOTED_SPREAD).all()
assert liquid_contracts["quoted_spread_dollars"].ge(0).all()

liquid_contracts.to_csv(LIQUID_OUTPUT, index=False)
print(
    f"Retained {len(liquid_contracts):,} of {len(research_contracts):,} "
    "research contracts after the $10k / 5¢ screen."
)
print(f"Saved liquid universe to {LIQUID_OUTPUT.resolve()}")

In [ ]:
liquid_bucket_counts = (
    liquid_contracts["research_bucket"].value_counts().rename("contracts")
)
liquidity_summary = liquid_contracts[
    ["quoted_spread_dollars", "yes_ask_depth_dollars", "yes_bid_depth_dollars"]
].describe(percentiles=[0.25, 0.5, 0.75, 0.9])

print("Retained contracts by bucket:")
print(liquid_bucket_counts.to_string())
print("\nLiquidity diagnostics:")
print(liquidity_summary.to_string())

preview_columns = [
    "research_bucket",
    "event_category",
    "event_title",
    "title",
    "ticker",
    "yes_bid_dollars",
    "yes_ask_dollars",
    "quoted_spread_dollars",
    "yes_ask_depth_dollars",
    "yes_bid_depth_dollars",
    "volume_fp",
    "open_interest_fp",
    "close_time",
]
liquid_contracts[preview_columns].head(25)

## Liquidity-screen result

The $10,000 YES-ask depth and ≤$0.05 spread screen retains **41 of 26,224** research contracts:

- Policy: **28**
- Macro: **13**
- Single-industry: **0**
- Geopolitical: **0**

Among retained contracts, the median quoted spread is **$0.01** and median displayed YES ask depth is approximately **$18,145**.

Output: `data/kalshi_active_contracts_liquid_10k_5c.csv`